In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "backend"))

from app.core.config import get_settings
from app.ranking.data import DataLoader
from app.ranking.evaluation import run_evaluation

In [ ]:
import pandas as pd

settings = get_settings()
loader = DataLoader(settings.data_perfume_dir, settings.data_organ_dir if settings.data_organ_dir.exists() else None)

summary = run_evaluation(loader, test_ratio=0.2, top_n=10, seed=42)
methods = [k for k in summary if isinstance(summary[k], dict)]
df = pd.DataFrame({m: summary[m] for m in methods})
print(df.T.to_string())
print(f"\ntest sessions: {summary.get('n_test_sessions')}, catalog: {summary.get('catalog_size')}")

## Анализ ошибок

Для каждой тестовой сессии смотрим, где оказался target и почему промах:
- **low_overlap** — мало общих нот между профилем и целевым SKU
- **weak_signal** — ноты совпадают, но слишком много похожих кандидатов
- **empty_profile** — пустой профиль

In [ ]:
import numpy as np
from app.ranking.profile.build_profile import session_to_user_vector
from app.ranking.scoring.score import build_sku_vectors, score_skus

perfume_notes = loader.load_perfume_notes()
sessions = loader.load_organ_sessions()
perfume_vectors, note_to_idx, idx_to_note = build_sku_vectors(perfume_notes)

sessions_shuffled = sessions.sample(frac=1, random_state=42)
n_test = max(1, int(len(sessions_shuffled) * 0.2))
test_sessions = sessions_shuffled.iloc[-n_test:]

rows = []
for _, row in test_sessions.iterrows():
    sid, target = int(row["session_id"]), int(row["target_perfume_id"])
    u = session_to_user_vector(sid, loader)
    if not u:
        rows.append({"sid": sid, "target": target, "hit": 0, "rank": None, "overlap": 0, "reason": "empty_profile"})
        continue
    recs = score_skus(u, perfume_vectors, note_to_idx, idx_to_note, top_n=100)
    pred = [r[0] for r in recs]
    rank = (pred.index(target) + 1) if target in pred else None
    hit = 1.0 if target in pred[:10] else 0.0

    u_notes = {k for k, v in u.items() if v > 0}
    t_notes = set(perfume_notes[perfume_notes["perfume_id"] == target]["note"].str.strip().str.lower())
    overlap = len(u_notes & t_notes)
    reason = "hit" if hit else ("low_overlap" if overlap <= 1 else "weak_signal")
    rows.append({"sid": sid, "target": target, "hit": hit, "rank": rank, "overlap": overlap, "reason": reason})

err_df = pd.DataFrame(rows)
fails = err_df[err_df["hit"] == 0]
print(f"Hit@10: {err_df['hit'].mean():.3f}")
print(f"Misses: {len(fails)}/{len(err_df)} ({100*len(fails)/len(err_df):.0f}%)")
print(fails["reason"].value_counts().to_string())

found = err_df[err_df["rank"].notna()]
if len(found):
    print(f"\nMedian rank: {found['rank'].median():.0f}, mean: {found['rank'].mean():.1f}")